# Speech-Based Cognitive Screening — Notebook 03
## The XAI Faithfulness Audit · Do SHAP and LIME tell the truth?

**Team:** A. R. Nitesh Kumar (23BAI0108) · Harshini J. S. (23BAI0166) · Hari Prasath U. (23BAI0038)  


**What this notebook does — the Explainable-AI contribution.**  
Notebook 02 built a concept bottleneck whose linear head gives the **exact** contribution of each
concept: `contributionⱼ = weightⱼ × conceptⱼ`, summing to the logit. That gives us something almost
no clinical study has — a **ground-truth explanation**.

So we can finally ask the question everyone skips:

> **When people run SHAP and LIME on a clinical speech model, do those explanations actually match
> what the model is really doing — or do they mislead?**

We run SHAP and LIME on the bottleneck, then score them against the exact truth with three metrics:
Spearman rank correlation, sign agreement, and top-3 overlap. No new data, no audio, no GPU.


## 0 · Setup

In [ ]:
!pip install torch scikit-learn shap lime pandas pyarrow scipy --quiet
import numpy as np, pandas as pd, torch, torch.nn as nn, warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from scipy.stats import spearmanr
torch.manual_seed(0); np.random.seed(0)
print("setup OK")

## 1 · Load features & rebuild the bottleneck

Upload **`pitt_cookie_features.csv`** from Notebook 01. We retrain the same concept bottleneck here
so this notebook is self-contained.


In [ ]:
import glob
cand = glob.glob("**/pitt_cookie_features.*", recursive=True)
if not cand:
    try:
        from google.colab import files
        print("Select pitt_cookie_features.csv …"); up = files.upload(); cand = list(up.keys())
    except Exception as e: print("upload skipped:", e)
assert cand, "Upload pitt_cookie_features.csv and re-run."
path = ([c for c in cand if c.endswith(".parquet")] or cand)[0]
df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)

d = df[df.dx.isin(["Control","ProbableAD"])].copy()
d["label"] = (d.dx=="ProbableAD").astype(int)
CONCEPTS = ["mattr","pronoun_noun_ratio","mlu_words","lexical_density","filled_pause_rate",
            "speech_rate_wpm","long_pause_ratio","mean_pause_ms","ttr","noun_verb_ratio","pronoun_ratio"]
k = len(CONCEPTS)
X = d[CONCEPTS].values.astype(np.float32); y = d["label"].values.astype(np.float32)
imp = SimpleImputer().fit(X); sc = StandardScaler().fit(imp.transform(X))
Xs = sc.transform(imp.transform(X)).astype(np.float32)
print(f"{len(d)} recordings · {k} concepts")

In [ ]:
class Bottleneck(nn.Module):
    def __init__(self, k):
        super().__init__()
        self.g = nn.Sequential(nn.Linear(k,32), nn.ReLU(), nn.Linear(32,k))
        self.head = nn.Linear(k,1)
    def forward(self, x):
        c = self.g(x); return self.head(c).squeeze(-1), c

m = Bottleneck(k)
opt = torch.optim.Adam(m.parameters(), lr=0.01, weight_decay=1e-4)
bce, mse = nn.BCEWithLogitsLoss(), nn.MSELoss()
Xt, yt = torch.tensor(Xs), torch.tensor(y)
for _ in range(300):
    m.train(); opt.zero_grad(); lo, c = m(Xt)
    (bce(lo, yt) + 0.5*mse(c, Xt)).backward(); opt.step()
m.eval()
W = m.head.weight.detach().numpy().ravel()
print("bottleneck trained · head weights ready")

## 2 · The ground truth — exact attributions

For every recording, the true contribution of each concept is `weight × concept_value`. This is not
an estimate; it is what the model literally computes. Everything else is judged against this.


In [ ]:
with torch.no_grad():
    _, C_all = m(Xt)
C_all = C_all.numpy()
GT = C_all * W                      # exact per-concept contribution, shape (n, k)
print("ground-truth attribution matrix:", GT.shape)

# model as a plain function for SHAP/LIME
def f_np(a):
    a = torch.tensor(a, dtype=torch.float32)
    with torch.no_grad(): lo, _ = m(a)
    return torch.sigmoid(lo).numpy()

## 3 · Run SHAP

KernelSHAP estimates each concept's contribution using a background sample. We audit 30 recordings.


In [ ]:
import shap
rng = np.random.RandomState(0)
idxs = rng.choice(len(Xs), 30, replace=False)
bg = shap.sample(Xs, 40, random_state=0)
explainer = shap.KernelExplainer(f_np, bg)
SH = np.array(explainer.shap_values(Xs[idxs], nsamples=100, silent=True))
print("SHAP attributions:", SH.shape)

## 4 · Run LIME

LIME fits a local linear surrogate around each recording and reports its coefficients as the
explanation.


In [ ]:
from lime.lime_tabular import LimeTabularExplainer
limex = LimeTabularExplainer(Xs, feature_names=CONCEPTS, mode="regression",
                             discretize_continuous=False, random_state=0)
LI = []
for i in idxs:
    e = limex.explain_instance(Xs[i], f_np, num_features=k)
    w = dict(e.as_list())
    LI.append([w.get(cn, 0.0) for cn in CONCEPTS])
LI = np.array(LI)
print("LIME attributions:", LI.shape)

## 5 · Score faithfulness against the exact truth

Three complementary metrics, each in [−1, 1] or [0, 1]:
- **Spearman** — does the method rank the concepts in the right order? (1 = perfect, 0 = random, <0 = reversed)
- **Sign agreement** — does it get the *direction* (toward AD vs Control) right?
- **Top-3 overlap** — do its 3 most-important concepts match the true 3?


In [ ]:
def metrics(A):
    rhos, signs, top3 = [], [], []
    for j, i in enumerate(idxs):
        g, a = GT[i], A[j]
        rho = spearmanr(g, a).correlation
        rhos.append(0 if np.isnan(rho) else rho)
        signs.append(np.mean(np.sign(g) == np.sign(a)))
        tg = set(np.argsort(-np.abs(g))[:3]); ta = set(np.argsort(-np.abs(a))[:3])
        top3.append(len(tg & ta)/3)
    return np.mean(rhos), np.mean(signs), np.mean(top3)

print("=== FAITHFULNESS vs EXACT ground truth (30 recordings) ===\n")
print(f"{'method':8s} {'Spearman':>10s} {'sign-agree':>12s} {'top3-overlap':>14s}")
for nm, A in [("SHAP", SH), ("LIME", LI)]:
    r, s, t = metrics(A)
    print(f"{nm:8s} {r:10.3f} {s:12.3f} {t:14.3f}")
print("\n1.0 = perfectly faithful to the model's true reasoning.")
print("Values well below 1 mean the tool's explanation does NOT reflect what the model actually did —")
print("a direct warning for anyone trusting SHAP/LIME on clinical speech models.")

## 6 · See it on one recording

Put the exact truth next to what SHAP and LIME claim, for a single patient. Where the ranked lists
disagree, the post-hoc tool would have told the clinician the wrong reason.


In [ ]:
j = 0; i = idxs[j]
def top_names(vec, n=5):
    return [(CONCEPTS[t], round(float(vec[t]), 3)) for t in np.argsort(-np.abs(vec))[:n]]

print(f"Recording {d.iloc[i]['file']} (true label {int(y[i])}: {'AD' if y[i]==1 else 'Control'})\n")
print("EXACT truth (top 5):")
for nm, v in top_names(GT[i]): print(f"   {nm:20s} {v:+.3f}")
print("\nSHAP says (top 5):")
for nm, v in top_names(SH[j]): print(f"   {nm:20s} {v:+.3f}")
print("\nLIME says (top 5):")
for nm, v in top_names(LI[j]): print(f"   {nm:20s} {v:+.3f}")

## 7 · What you produced & what's next

- A **quantitative faithfulness audit** of SHAP and LIME on a clinical speech model, scored against
  ground truth that is exact rather than assumed — the core Explainable-AI result, and the part of
  the project with the least prior work.
- Evidence for the paper's central claim: **post-hoc explanations on these models may be unfaithful**,
  so an intrinsically interpretable bottleneck is the safer design.

### The three course cores are now complete
- **Speech & Language (E1):** markers + baseline — Notebook 01
- **Deep Learning:** concept bottleneck + interpretability cost — Notebook 02
- **Explainable AI:** SHAP/LIME faithfulness audit — Notebook 03

### Sensible next steps
1. **Strengthen the story:** repeat the audit across seeds / more recordings and report mean ± std;
   add Grad-CAM once the audio CNN exists.
2. **Experiment E2 (audio):** now fetch the MP3s and add acoustic concepts (pitch, jitter, HNR,
   spectrogram CNN, wav2vec2), then re-run Notebooks 02–03 with the enlarged concept set.
3. **MCI task:** rerun all three notebooks on the harder MCI-vs-Control problem.

> These faithfulness numbers move with sample size and the SHAP background — always report them as
> **mean ± std over several runs**, never a single value.
